In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn

In [2]:
TRAIN_FEATURE_PATH = Path(
    "/kaggle/input/notebooks/zidanefatuna/09-notebook-image-only-baseline/"
    "baseline_features/train_features.npz"
)

VAL_FEATURE_PATH = TRAIN_FEATURE_PATH.parent / "validation_features.npz"

with np.load(TRAIN_FEATURE_PATH, allow_pickle=False) as train_cache:
    X_train = train_cache["features"]
    y_train = train_cache["labels"]
    train_ids = train_cache["study_ids"]
    target_columns = train_cache["target_columns"]
    planes = train_cache["planes"]

with np.load(VAL_FEATURE_PATH, allow_pickle=False) as val_cache:
    X_val = val_cache["features"]
    y_val = val_cache["labels"]
    val_ids = val_cache["study_ids"]

    assert np.array_equal(
        target_columns, val_cache["target_columns"]
    )
    assert np.array_equal(planes, val_cache["planes"])

X = np.concatenate([X_train, X_val]).astype(np.float32)
y = np.concatenate([y_train, y_val]).astype(np.float32)
study_ids = np.concatenate([train_ids, val_ids])

TARGET_COLUMNS = target_columns.tolist()

assert X.shape == (58, 1536)
assert y.shape == (58, 12)
assert len(np.unique(study_ids)) == 58
assert np.isfinite(X).all()
assert np.isin(y, [0, 1]).all()

print("Features:", X.shape)
print("Labels:", y.shape)
print("Unique studies:", len(np.unique(study_ids)))

Features: (58, 1536)
Labels: (58, 12)
Unique studies: 58


In [3]:
rng = np.random.default_rng(42)
shuffled_indices = rng.permutation(len(study_ids))
fold_indices = np.array_split(shuffled_indices, 5)

fold_assignment = np.full(len(study_ids), -1, dtype=int)

for fold, indices in enumerate(fold_indices):
    fold_assignment[indices] = fold

assert (fold_assignment >= 0).all()

fold_manifest = pd.DataFrame({
    "StudyInstanceUID": study_ids,
    "fold": fold_assignment,
})

print("Studies per evaluation fold:")
print(fold_manifest["fold"].value_counts().sort_index())

Studies per evaluation fold:
fold
0    12
1    12
2    12
3    11
4    11
Name: count, dtype: int64


In [4]:
coverage_records = []

for fold in range(5):
    evaluation_mask = fold_assignment == fold
    training_mask = ~evaluation_mask

    for j, target in enumerate(TARGET_COLUMNS):
        coverage_records.append({
            "fold": fold,
            "target": target,
            "train_positive": int(y[training_mask, j].sum()),
            "train_negative": int((y[training_mask, j] == 0).sum()),
            "eval_positive": int(y[evaluation_mask, j].sum()),
            "eval_negative": int((y[evaluation_mask, j] == 0).sum()),
        })

fold_coverage = pd.DataFrame(coverage_records)

display(
    fold_coverage.pivot(
        index="target",
        columns="fold",
        values="eval_positive",
    )
)

print("Fold–target combinations missing a class:")
display(
    fold_coverage.loc[
        fold_coverage[
            ["train_positive", "train_negative",
             "eval_positive", "eval_negative"]
        ].eq(0).any(axis=1)
    ]
)

fold,0,1,2,3,4
target,,,,,
ACL,5,5,6,4,4
Baker's,4,2,0,2,4
Contusion,3,7,4,5,0
Effusion,8,7,8,7,5
Fracture,4,3,6,3,2
Lateral Meniscus,7,3,3,6,4
Lateral OA,7,1,0,1,2
MCL,2,2,3,2,0
Medial Meniscus,4,6,3,7,6


Fold–target combinations missing a class:


,fold,target,train_positive,train_negative,eval_positive,eval_negative
29,2,Lateral OA,11,35,0,12
33,2,Baker's,12,34,0,12
49,4,MCL,9,38,0,11
58,4,Contusion,19,28,0,11


In [5]:
import copy


def prepare_features(train_x, other_x):
    mean = train_x.mean(axis=0, keepdims=True)
    std = train_x.std(axis=0, keepdims=True)
    std = np.where(std < 1e-6, 1.0, std)

    return (
        torch.from_numpy((train_x - mean) / std).float(),
        torch.from_numpy((other_x - mean) / std).float(),
    )


def initialize_classifier(training_labels):
    model = nn.Linear(X.shape[1], y.shape[1])

    prevalence = np.clip(
        training_labels.mean(axis=0), 1e-6, 1 - 1e-6
    )

    with torch.no_grad():
        model.weight.zero_()
        model.bias.copy_(
            torch.from_numpy(np.log(prevalence / (1 - prevalence)))
        )

    return model, prevalence


def train_step(model, optimizer, features, targets):
    model.train()
    optimizer.zero_grad()

    loss = nn.functional.binary_cross_entropy_with_logits(
        model(features), targets
    )
    loss.backward()
    optimizer.step()


def evaluation_bce(model, features, targets):
    model.eval()
    with torch.no_grad():
        return nn.functional.binary_cross_entropy_with_logits(
            model(features), targets
        ).item()

In [6]:
cv_records = []
oof_predictions = np.full(y.shape, np.nan, dtype=np.float32)
oof_reference = np.full(y.shape, np.nan, dtype=np.float32)

for fold in range(5):
    torch.manual_seed(42 + fold)

    outer_train = np.flatnonzero(fold_assignment != fold)
    outer_eval = np.flatnonzero(fold_assignment == fold)

    # Inner split uses outer-training studies only.
    inner_rng = np.random.default_rng(100 + fold)
    inner_order = inner_rng.permutation(outer_train)
    n_inner_val = int(np.ceil(0.2 * len(inner_order)))

    inner_val = inner_order[:n_inner_val]
    inner_train = inner_order[n_inner_val:]

    inner_x, inner_val_x = prepare_features(
        X[inner_train], X[inner_val]
    )
    inner_y = torch.from_numpy(y[inner_train])
    inner_val_y = torch.from_numpy(y[inner_val])

    model, _ = initialize_classifier(y[inner_train])
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=1e-3, weight_decay=0.1
    )

    selected_epoch = 0
    lowest_inner_bce = evaluation_bce(
        model, inner_val_x, inner_val_y
    )

    for epoch in range(1, 101):
        train_step(model, optimizer, inner_x, inner_y)
        score = evaluation_bce(model, inner_val_x, inner_val_y)

        if score < lowest_inner_bce:
            lowest_inner_bce = score
            selected_epoch = epoch

    # Reinitialize and fit using all outer-training studies.
    outer_x, eval_x = prepare_features(
        X[outer_train], X[outer_eval]
    )
    outer_y = torch.from_numpy(y[outer_train])
    eval_y = torch.from_numpy(y[outer_eval])

    model, prevalence = initialize_classifier(y[outer_train])
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=1e-3, weight_decay=0.1
    )

    for _ in range(selected_epoch):
        train_step(model, optimizer, outer_x, outer_y)

    model.eval()
    with torch.no_grad():
        logits = model(eval_x)
        probabilities = torch.sigmoid(logits).numpy()
        model_bce = nn.functional.binary_cross_entropy_with_logits(
            logits, eval_y
        ).item()

    reference = np.repeat(
        prevalence[None, :], len(outer_eval), axis=0
    )
    reference_bce = -(
        y[outer_eval] * np.log(reference)
        + (1 - y[outer_eval]) * np.log(1 - reference)
    ).mean()

    oof_predictions[outer_eval] = probabilities
    oof_reference[outer_eval] = reference

    cv_records.append({
        "fold": fold,
        "training_studies": len(outer_train),
        "evaluation_studies": len(outer_eval),
        "selected_epoch": selected_epoch,
        "reference_bce": float(reference_bce),
        "model_bce": model_bce,
        "improvement": float(reference_bce - model_bce),
    })

    print(
        f"Fold {fold}: epoch {selected_epoch}, "
        f"reference BCE {reference_bce:.4f}, "
        f"model BCE {model_bce:.4f}"
    )

cv_results = pd.DataFrame(cv_records)

assert np.isfinite(oof_predictions).all()
assert np.isfinite(oof_reference).all()

display(cv_results.round(4))

Fold 0: epoch 3, reference BCE 0.7269, model BCE 0.6905
Fold 1: epoch 1, reference BCE 0.6003, model BCE 0.5924
Fold 2: epoch 4, reference BCE 0.5893, model BCE 0.6021
Fold 3: epoch 0, reference BCE 0.6392, model BCE 0.6392
Fold 4: epoch 3, reference BCE 0.6189, model BCE 0.5720


,fold,training_studies,evaluation_studies,selected_epoch,reference_bce,model_bce,improvement
0,0,46,12,3,0.7269,0.6905,0.0364
1,1,46,12,1,0.6003,0.5924,0.0079
2,2,46,12,4,0.5893,0.6021,-0.0128
3,3,47,11,0,0.6392,0.6392,0.0000
4,4,47,11,3,0.6189,0.5720,0.0469


In [7]:
from sklearn.metrics import roc_auc_score

auc_results = pd.DataFrame({
    "target": TARGET_COLUMNS,
    "model_auc": [
        roc_auc_score(y[:, j], oof_predictions[:, j])
        for j in range(len(TARGET_COLUMNS))
    ],
})

display(auc_results.round(4))
print("Macro OOF AUC:", round(auc_results["model_auc"].mean(), 4))

,target,model_auc
0,ACL,0.4412
1,MCL,0.3946
2,Medial Meniscus,0.5877
3,Lateral Meniscus,0.6373
4,Medial OA,0.6620
5,Lateral OA,0.4313
6,PF OA,0.6281
7,Effusion,0.6099
8,Synovitis,0.5615
9,Baker's,0.4511


Macro OOF AUC: 0.5262


In [8]:
fold_auc_records = []

for fold in range(5):
    mask = fold_assignment == fold

    for j, target in enumerate(TARGET_COLUMNS):
        labels = y[mask, j]

        auc = (
            roc_auc_score(labels, oof_predictions[mask, j])
            if np.unique(labels).size == 2
            else np.nan
        )

        fold_auc_records.append({
            "fold": fold,
            "target": target,
            "auc": auc,
        })

fold_auc_results = pd.DataFrame(fold_auc_records)

display(
    fold_auc_results.groupby("fold").agg(
        macro_auc=("auc", "mean"),
        evaluable_targets=("auc", "count"),
    ).round(4)
)

,macro_auc,evaluable_targets
fold,,
0,0.5958,12
1,0.5327,12
2,0.5122,10
3,0.5000,12
4,0.6942,10


In [9]:
CV_DIR = Path("/kaggle/working/cross_validation_results")
CV_DIR.mkdir(parents=True, exist_ok=True)

cv_results.to_csv(CV_DIR / "fold_bce_results.csv", index=False)
auc_results.to_csv(CV_DIR / "pooled_target_auc.csv", index=False)
fold_manifest.to_csv(CV_DIR / "study_folds.csv", index=False)

np.savez_compressed(
    CV_DIR / "oof_predictions.npz",
    study_ids=study_ids,
    labels=y,
    predictions=oof_predictions,
    reference_predictions=oof_reference,
    folds=fold_assignment,
    target_columns=np.asarray(TARGET_COLUMNS, dtype=str),
)

print("Recorded pooled macro OOF AUC:",
      round(auc_results["model_auc"].mean(), 4))

Recorded pooled macro OOF AUC: 0.5262


In [10]:
def binary_cross_entropy(labels, probabilities):
    p = np.clip(probabilities.astype(np.float64), 1e-7, 1 - 1e-7)
    return float(
        -(labels * np.log(p) + (1 - labels) * np.log(1 - p)).mean()
    )

model_oof_bce = binary_cross_entropy(y, oof_predictions)
reference_oof_bce = binary_cross_entropy(y, oof_reference)

summary = pd.DataFrame([{
    "model_oof_bce": model_oof_bce,
    "reference_oof_bce": reference_oof_bce,
    "bce_improvement": reference_oof_bce - model_oof_bce,
    "pooled_macro_auc": auc_results["model_auc"].mean(),
}])

display(summary.round(4))
summary.to_csv(CV_DIR / "baseline_summary.csv", index=False)

,model_oof_bce,reference_oof_bce,bce_improvement,pooled_macro_auc
0,0.6197,0.6351,0.0154,0.5262
